# Regression

We fit linear models for yearly median GPU price using crypto, equity, and **GPU supply chain** returns (TSMC, NVIDIA, AMD, Micron). We compare: BTC only, BTC + S&P 500, BTC lagged, log-scale, and a full model with supply chain. Interpretation and model comparison below.

## Data and lag

In [15]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PROC = ROOT / "data" / "processed"

In [16]:
df = pd.read_csv(DATA_PROC / "merged_yearly.csv")
df["btc_avg_price_lag1"] = df["btc_avg_price"].shift(1)
import numpy as np
df["log_btc"] = np.log10(df["btc_avg_price"].clip(lower=1))
df["log_gpu_price"] = np.log10(df["median_gpu_price"].clip(lower=1))
sc_returns = [c for c in ["tsmc_return", "nvda_return", "amd_return", "micron_return"] if c in df.columns]
base_subset = ["median_gpu_price", "btc_avg_price", "sp500_return", "btc_avg_price_lag1", "log_btc", "log_gpu_price"]
df = df.dropna(subset=base_subset + (sc_returns if sc_returns else []))
y = df["median_gpu_price"]
y_log = df["log_gpu_price"]

## Model 1: BTC only

In [17]:
X1 = sm.add_constant(df["btc_avg_price"])
m1 = sm.OLS(y, X1).fit()
print(m1.summary2().tables[1])
print(f"R² = {m1.rsquared:.4f}")

                     Coef.    Std.Err.         t     P>|t|      [0.025  \
const          2642.927782  716.876925  3.686725  0.006159  989.806629   
btc_avg_price    -0.023484    0.016552 -1.418810  0.193717   -0.061653   

                    0.975]  
const          4296.048936  
btc_avg_price     0.014685  
R² = 0.2010


## Model 2: BTC + SP500 return

In [18]:
X2 = sm.add_constant(df[["btc_avg_price", "sp500_return"]])
m2 = sm.OLS(y, X2).fit()
print(m2.summary2().tables[1])
print(f"R² = {m2.rsquared:.4f}")

                     Coef.     Std.Err.         t     P>|t|        [0.025  \
const          3021.749764   673.983380  4.483419  0.002855   1428.032318   
btc_avg_price    -0.013731     0.015755 -0.871529  0.412356     -0.050984   
sp500_return  -6085.844635  3465.642581 -1.756051  0.122507 -14280.787130   

                    0.975]  
const          4615.467209  
btc_avg_price     0.023523  
sp500_return   2109.097860  
R² = 0.4454


## Model 3: BTC lagged one year

In [19]:
X3 = sm.add_constant(df["btc_avg_price_lag1"])
m3 = sm.OLS(y, X3).fit()
print(m3.summary2().tables[1])
print(f"R² = {m3.rsquared:.4f}")

                          Coef.    Std.Err.         t     P>|t|      [0.025  \
const               2190.818414  778.629233  2.813686  0.022714  395.296182   
btc_avg_price_lag1    -0.012460    0.026822 -0.464539  0.654643   -0.074312   

                         0.975]  
const               3986.340646  
btc_avg_price_lag1     0.049392  
R² = 0.0263


## Model 4: log GPU price ~ log BTC

Log-scale can improve fit when both variables span orders of magnitude.

In [20]:
X4 = sm.add_constant(df["log_btc"])
m4 = sm.OLS(y_log, X4).fit()
print(m4.summary2().tables[1])
print(f"R² = {m4.rsquared:.4f}")

            Coef.  Std.Err.         t     P>|t|    [0.025    0.975]
const    3.449630  0.780511  4.419707  0.002227  1.649768  5.249491
log_btc -0.091451  0.190551 -0.479931  0.644124 -0.530863  0.347960
R² = 0.0280


## Model 5: adding supply chain (TSMC, NVIDIA, AMD, Micron)

Supply chain returns proxy for demand and capacity: TSMC (fab), NVIDIA and AMD (design/vendor), Micron (memory). Adding them can improve fit if GPU pricing moves with the chain.

In [21]:
if sc_returns:
    X5 = sm.add_constant(df[["btc_avg_price", "sp500_return"] + sc_returns])
    m5 = sm.OLS(y, X5).fit()
    print(m5.summary2().tables[1])
    print(f"R² = {m5.rsquared:.4f}")
else:
    m5 = None

## Interpretation

- **BTC alone** explains little (low R²); same-year BTC is a weak linear predictor of median GPU price.
- **Adding S&P 500 return** often improves fit (equity and GPU demand can align).
- **Supply chain model (Model 5)** uses TSMC, NVIDIA, AMD, Micron returns; when available, it typically raises R² because GPU pricing tracks vendor and fab performance. Check coefficients: positive NVDA/AMD return coefficients suggest higher GPU prices when designer stocks do well.
- **Caveats:** Few years, so overfitting risk; correlation is not causation; yearly aggregation hides within-year volatility.

## Compare

Higher R² = more variance in GPU price explained. Log-scale (model 4) often does better when BTC and price range widely.

In [22]:
models = [("BTC", m1), ("BTC + SP500", m2), ("BTC lag1", m3), ("log GPU ~ log BTC", m4)]
if m5 is not None:
    models.append(("+ Supply chain", m5))
pd.DataFrame({
    "Model": [m[0] for m in models],
    "R²": [m[1].rsquared for m in models],
    "AIC": [m[1].aic for m in models],
})

,Model,R²,AIC
0,BTC,0.201040,178.398822
1,BTC + SP500,0.445371,176.748707
2,BTC lag1,0.026266,180.377100
3,log GPU ~ log BTC,0.027986,16.054890
